In [29]:
from hierarchicalcausalmodels.models import HSCMParametric
from causalgraphicalmodels import CausalGraphicalModel

In [30]:
def collapse(HSCMParametric : HSCMParametric) -> CausalGraphicalModel:
    """
    Collapse a hierarchical structural causal model (HSCM) into a non-hierarchical CGM.

    This function takes an HSCMParametric object and collapses its hierarchical structure,
    resulting in a collapsed flat causal graphical model that retains the causal relationships of the original model.

    Parameters
    ----------
    HSCMParametric : HSCMParametric
        The hierarchical structural causal model to be collapsed.

    Returns
    -------
    CausalGraphicalModel
        A collapsed flat causal graphical model.
    """
    nodes = HSCMParametric.unit_nodes.copy()
    edges = HSCMParametric.edges.copy()
    for subunit in HSCMParametric.subunit_nodes:
        q_node = "Q" + subunit + "| pa(" + subunit + ")"
        nodes.add(q_node)
        for parent, child  in HSCMParametric.edges :
            if child == subunit and parent in nodes :
                edges.remove((parent, child))
                edges.add((parent, q_node))
            elif parent == subunit and child in nodes :
                edges.remove((parent, child))
                edges.add((q_node, child))
            elif parent == subunit:
                edges.remove((parent, child))
        collapsed_model = CausalGraphicalModel(nodes=list(nodes), edges=list(edges))
    return collapsed_model

In [31]:
# Build three example HSCMs (confounder, confounder + interferer, instrument)
# and collapse each using the collapse() function.

def _empty_fun(*args, **kwargs):
    return None

# (a) Confounder: U -> A, U -> Y, A -> Y (A, Y are subunit-level)
hscm_confounder = HSCMParametric(
    nodes={"U", "A", "Y"},
    edges={("U", "A"), ("U", "Y"), ("A", "Y")},
    unit_nodes={"U"},
    subunit_nodes={"A", "Y"},
    sizes=[3],
    node_functions={"U": _empty_fun, "A": _empty_fun, "Y": _empty_fun},
    data={},
)
confounder_cgm = collapse(hscm_confounder)
print("Confounder CGM nodes:", confounder_cgm.dag.nodes)
print("Confounder CGM edges:", confounder_cgm.dag.edges)

# (b) Confounder & Interferer: U -> A, U -> Y, A -> Y, A -> Z, Z -> Y
# A, Y are subunit-level; Z, U are unit-level.
hscm_confounder_interferer = HSCMParametric(
    nodes={"U", "Z", "A", "Y"},
    edges={("U", "A"), ("U", "Y"), ("A", "Y"), ("A", "Z"), ("Z", "Y")},
    unit_nodes={"U", "Z"},
    subunit_nodes={"A", "Y"},
    sizes=[3],
    node_functions={"U": _empty_fun, "Z": _empty_fun, "A": _empty_fun, "Y": _empty_fun},
    data={},
)
confounder_interferer_cgm = collapse(hscm_confounder_interferer)
print("Confounder Interferer CGM nodes:", confounder_interferer_cgm.dag.nodes)
print("Confounder Interferer CGM edges:", confounder_interferer_cgm.dag.edges)


# (c) Instrument: U -> A, U -> Y, Z -> A, A -> Y
# Z, A are subunit-level; U, Y are unit-level.
hscm_instrument = HSCMParametric(
    nodes={"U", "Y", "Z", "A"},
    edges={("U", "A"), ("U", "Y"), ("Z", "A"), ("A", "Y")},
    unit_nodes={"U", "Y"},
    subunit_nodes={"Z", "A"},
    sizes=[3],
    node_functions={"U": _empty_fun, "Y": _empty_fun, "Z": _empty_fun, "A": _empty_fun},
    data={},
)
instrument_cgm = collapse(hscm_instrument)
print("Instrument CGM nodes:", instrument_cgm.dag.nodes)
print("Instrument CGM edges:", instrument_cgm.dag.edges) 



Confounder CGM nodes: ['U', 'Q_A| pa(_A)', 'Q_Y| pa(_Y)']
Confounder CGM edges: [('U', 'Q_Y| pa(_Y)'), ('U', 'Q_A| pa(_A)')]
Confounder Interferer CGM nodes: ['U', 'Q_A| pa(_A)', 'Q_Y| pa(_Y)', 'Z']
Confounder Interferer CGM edges: [('U', 'Q_A| pa(_A)'), ('U', 'Q_Y| pa(_Y)'), ('Q_A| pa(_A)', 'Z'), ('Z', 'Q_Y| pa(_Y)')]
Instrument CGM nodes: ['U', 'Q_A| pa(_A)', 'Y', 'Q_Z| pa(_Z)']
Instrument CGM edges: [('U', 'Y'), ('U', 'Q_A| pa(_A)'), ('Q_A| pa(_A)', 'Y')]


In [33]:
# I'll just write mechanisms but we can get it from HSCMParametric.functions by restraining to unit_level variables
# we can represent each mecanism as a dictionnary containing the function for sampling, the parents in the order the function should take them and finally a string expressing whether or not another function is present in there

def augment_collapsed_model(collapsed_model, q_hat, q_hat_expr, mechanisms) -> CausalGraphicalModel:
    """
    Augment a collapsed causal graphical model with a new node representing a subunit variable.

    This function adds a new node to the collapsed model, representing a subunit variable
    conditioned on its parents. It also updates the edges and mechanisms accordingly.

    Parameters
    ----------
    CausalGraphicalModel : collapsed_model
        The collapsed causal graphical model to be augmented.
    str : q_hat
        The name of the new node to be added.
    set : q_hat_parents
        The set of parent nodes for the new node.
    function : q_hat_expr
        The functional expression defining the new node.
    dict : mechanisms
        A dictionary of mechanisms for the nodes in the model.

    Returns
    -------
    CausalGraphicalModel
        The augmented causal graphical model.
    """
    node = q_hat
    collapsed_model.add_node(node)
    mechanisms[node] = q_hat_expr
    for parent in q_hat_expr['parents']:
        collapsed_model.add_edge(parent, node)
        collapsed_model.add_edge(node,parent)
    for unit in collapsed_model.dag.nodes:
        if q_hat_expr["expr"] in mechanisms[unit]["expr"]:
            collapsed_model.add_edge(q_hat, unit)
            mechanisms[unit]["expr"] = mechanisms[unit]["expr"].replace(q_hat_expr["expr"], q_hat)
            mechanisms[unit]["parents"].add(q_hat)
            for parent in q_hat_expr['parents']:
                if parent not in mechanisms[unit]["expr"]:
                    if (parent, unit) in collapsed_model.dag.edges:
                        collapsed_model.remove_edge(parent, unit)
                        mechanisms[unit]["parents"].remove(parent)
    return collapsed_model
        
    
    